In [0]:
CREATE TABLE if not EXISTS gold.fact_shipments
(
    shipment_id INT,

    order_id INT,

    customer_sk BIGINT,

    carrier STRING,

    tracking_number STRING,

    shipment_status STRING,

    shipped_date TIMESTAMP,

    delivered_date TIMESTAMP,

    created_at TIMESTAMP,

    updated_at TIMESTAMP,

    created_timestamp TIMESTAMP,

    updated_timestamp TIMESTAMP
)
USING DELTA;

In [0]:
MERGE INTO de_learning_workspace.gold.fact_shipments AS target

USING
(
    SELECT

        s.shipment_id,

        s.order_id,

        c.customer_sk,

        s.carrier,

        s.tracking_number,

        s.shipment_status,

        s.shipped_date,

        s.delivered_date,

        s.created_at,

        s.updated_at

    FROM de_learning_workspace.silver.shipments s


    JOIN de_learning_workspace.silver.orders o
    ON s.order_id = o.order_id


    JOIN de_learning_workspace.gold.dim_customer c
    ON o.customer_id = c.customer_id
    AND c.is_current = true

) AS source


ON target.shipment_id = source.shipment_id


WHEN MATCHED THEN

UPDATE SET

    target.carrier = source.carrier,

    target.tracking_number = source.tracking_number,

    target.shipment_status = source.shipment_status,

    target.shipped_date = source.shipped_date,

    target.delivered_date = source.delivered_date,

    target.updated_at = source.updated_at,

    target.updated_timestamp = current_timestamp()


WHEN NOT MATCHED THEN

INSERT
(
    shipment_id,
    order_id,
    customer_sk,
    carrier,
    tracking_number,
    shipment_status,
    shipped_date,
    delivered_date,
    created_at,
    updated_at,
    created_timestamp,
    updated_timestamp
)

VALUES
(
    source.shipment_id,
    source.order_id,
    source.customer_sk,
    source.carrier,
    source.tracking_number,
    source.shipment_status,
    source.shipped_date,
    source.delivered_date,
    source.created_at,
    source.updated_at,
    current_timestamp(),
    current_timestamp()
);

In [0]:
%python
print("Gold Layer Fact Shipments Loaded Successfully")

In [0]:
select * from de_learning_workspace.gold.fact_shipments;